# Adobe 10-K RAG: Bi-Encoder vs Cross-Encoder Reranking

This notebook runs the end-to-end RAG experiment on Kaggle:

1. Load the OCR-extracted Adobe 2025 Form 10-K corpus.
2. Build a dense bi-encoder index and retrieve the Top 10 chunks for each query.
3. Baseline pipeline: send the dense Top 3 chunks to the language model.
4. Reranked pipeline: cross-encode the same Top 10 query/chunk pairs, select the reranked Top 3, and send them to the same language model.
5. Compare rank movements, retrieved evidence, answers, and latency; save the results.

**Fair comparison:** both pipelines use the same dense Top 10 candidate pool, generation model, prompt template, and generation settings. Only the ordering/selection of the three context chunks changes.

> Before running: attach your private `adobe-extracted` dataset under Kaggle's **Add Input**. If you want generated answers, add a Kaggle Secret named `GEMINI_API_KEY`. Retrieval and reranking work without that secret.

## 0. Install dependencies

Run this cell once per Kaggle session. Internet must be enabled if the models are not already cached.

In [18]:
%pip install -q -U sentence-transformers transformers accelerate bitsandbytes pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


## 1. Configuration

The defaults use BGE base models, which are more substantial than the MiniLM alternatives and fit comfortably on most Kaggle T4 sessions. If GPU memory is tight, replace them with the commented MiniLM model IDs. Model IDs and prompt settings are kept in one place for reproducibility.

In [19]:
from pathlib import Path
import os
import re
import json
import time
import hashlib
import platform
import numpy as np
import pandas as pd
import torch

# Kaggle dataset path supplied for this project.
DATASET_ROOT = Path("/kaggle/input/datasets/vidurshah04/adobe-extracted")

# Output files are written to Kaggle's writable working directory.
OUTPUT_DIR = Path("/kaggle/working/rag_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# GPU-friendly, relatively strong default models.
EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-base"

# Lighter alternatives if you prefer faster, smaller models:
# EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
# RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

TOP_K_CANDIDATES = 10
TOP_K_CONTEXT = 3
CHUNK_SIZE_WORDS = 350
CHUNK_OVERLAP_WORDS = 60
BATCH_SIZE = 32

# Local Hugging Face generation: no API key required.
# Gemma 3 4B is a capable instruction-tuned model; 4-bit loading helps fit on a Kaggle T4.
GENERATION_MODEL_NAME = "google/gemma-3-4b-it"
GENERATION_MAX_NEW_TOKENS = 400
RUN_GENERATION = True
MAX_GENERATION_QUERIES = 6  # Two local generations per query.
LOAD_GENERATOR_IN_4BIT = True

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Dataset root exists:", DATASET_ROOT.exists(), "->", DATASET_ROOT)
print("Output directory:", OUTPUT_DIR)

Python: 3.13.15
PyTorch: 2.11.0+cu128
Device: cuda
GPU: Tesla T4
Dataset root exists: True -> /kaggle/input/datasets/vidurshah04/adobe-extracted
Output directory: /kaggle/working/rag_outputs


## 2. Locate and load the corpus

The loader looks for `chunks.json` first, then `pages.json`, then page-level `.txt` files. It searches the supplied root first and `/kaggle/input` second. If only a PDF is attached, it will try ordinary PDF text extraction but stop with a useful message if the embedded text is still corrupted; upload the OCR-extracted JSON/TXT files in that case.

In [20]:
def find_file(filename):
    """Find a named file, preferring the configured dataset root."""
    search_roots = [DATASET_ROOT, Path("/kaggle/input")]
    seen = set()
    for root in search_roots:
        if not root.exists():
            continue
        matches = sorted(root.rglob(filename))
        for match in matches:
            resolved = str(match.resolve())
            if resolved not in seen:
                seen.add(resolved)
                return match
    return None


def normalize_record(record, fallback_id=0, fallback_page=1):
    """Normalize a chunk/page dict into id, page, and text fields."""
    if isinstance(record, str):
        record = {"text": record}
    text = record.get("text", record.get("content", record.get("chunk", "")))
    if not isinstance(text, str):
        text = str(text)
    text = re.sub(r"(?im)^\s*Table of Contents\s*$", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    page_value = record.get("page", record.get("page_number", fallback_page))
    try:
        page_value = int(page_value)
    except (TypeError, ValueError):
        page_value = fallback_page
    chunk_id = record.get("chunk_id", record.get("id", fallback_id))
    return {"chunk_id": int(fallback_id), "source_id": str(chunk_id), "page": page_value, "text": text}


def build_chunks_from_pages(page_records, chunk_size=CHUNK_SIZE_WORDS, overlap=CHUNK_OVERLAP_WORDS):
    """Build word-window chunks within each source page to preserve page citations."""
    output = []
    for page_idx, rec in enumerate(page_records, start=1):
        page = normalize_record(rec, fallback_id=page_idx - 1, fallback_page=page_idx)
        words = page["text"].split()
        start = 0
        while start < len(words):
            end = min(start + chunk_size, len(words))
            chunk_text = " ".join(words[start:end]).strip()
            if len(chunk_text.split()) >= 40:
                output.append({
                    "chunk_id": len(output),
                    "source_id": f"page-{page['page']}-words-{start}-{end}",
                    "page": page["page"],
                    "text": chunk_text,
                })
            if end >= len(words):
                break
            start = max(0, end - overlap)
    return output


chunks_path = find_file("chunks.json")
pages_path = find_file("pages.json")

if chunks_path is not None:
    print("Loading existing chunks:", chunks_path)
    with open(chunks_path, "r", encoding="utf-8") as f:
        raw_chunks = json.load(f)
    if isinstance(raw_chunks, dict):
        raw_chunks = raw_chunks.get("chunks", raw_chunks.get("data", []))
    chunks = [normalize_record(rec, i, i + 1) for i, rec in enumerate(raw_chunks)]
    chunks = [c for c in chunks if len(c["text"].split()) >= 20]
    # Reassign stable sequential IDs after filtering.
    for i, chunk in enumerate(chunks):
        chunk["chunk_id"] = i
elif pages_path is not None:
    print("Loading pages and creating chunks:", pages_path)
    with open(pages_path, "r", encoding="utf-8") as f:
        raw_pages = json.load(f)
    if isinstance(raw_pages, dict):
        raw_pages = raw_pages.get("pages", raw_pages.get("data", []))
    pages = [normalize_record(rec, i, i + 1) for i, rec in enumerate(raw_pages)]
    chunks = build_chunks_from_pages(pages)
else:
    # Support datasets containing page_001.txt, page_002.txt, etc.
    txt_files = []
    for root in [DATASET_ROOT, Path("/kaggle/input")]:
        if root.exists():
            txt_files = sorted(root.rglob("page_*.txt"))
            if txt_files:
                break
    if txt_files:
        print(f"Loading {len(txt_files)} page-level text files")
        pages = []
        for i, path in enumerate(txt_files, start=1):
            match = re.search(r"page[_-]?(\d+)", path.stem, flags=re.I)
            page_no = int(match.group(1)) if match else i
            pages.append({"page": page_no, "text": path.read_text(encoding="utf-8", errors="replace")})
        chunks = build_chunks_from_pages(pages)
    else:
        pdf_candidates = []
        for root in [DATASET_ROOT, Path("/kaggle/input")]:
            if root.exists():
                pdf_candidates = sorted(root.rglob("*.pdf"))
                if pdf_candidates:
                    break
        if not pdf_candidates:
            raise FileNotFoundError(
                "Could not find chunks.json, pages.json, page_*.txt, or a PDF under /kaggle/input. "
                "Check that the adobe-extracted dataset is attached."
            )
        import fitz
        pdf_path = pdf_candidates[0]
        print("No extracted JSON/TXT found. Trying embedded PDF text:", pdf_path)
        pdf_doc = fitz.open(pdf_path)
        pages = [{"page": i + 1, "text": page.get_text("text")} for i, page in enumerate(pdf_doc)]
        all_text = " ".join(p["text"] for p in pages)
        alpha_ratio = sum(ch.isalpha() for ch in all_text) / max(1, len(all_text))
        if len(all_text) < 1000 or alpha_ratio < 0.35 or sum(ord(ch) < 9 for ch in all_text) > 20:
            raise RuntimeError(
                "PDF text extraction appears corrupted (as happened with this source PDF). "
                "Upload OCR-extracted pages.json or page_*.txt files, then rerun."
            )
        chunks = build_chunks_from_pages(pages)

if not chunks:
    raise ValueError("No usable text chunks were loaded. Inspect the uploaded corpus files.")

print(f"Loaded {len(chunks)} chunks")
print("Total words:", f"{sum(len(c['text'].split()) for c in chunks):,}")
print("Example metadata:", {k: chunks[0][k] for k in ["chunk_id", "page", "source_id"]})
print("Example text:", chunks[0]["text"][:500])

Loading pages and creating chunks: /kaggle/input/datasets/vidurshah04/adobe-extracted/pages.json
Loaded 205 chunks
Total words: 57,645
Example metadata: {'chunk_id': 0, 'page': 1, 'source_id': 'page-1-words-0-350'}
Example text: UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM 10-K (Mark One) x} ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fiscal year ended November 28, 2025 or TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the transition period from to Commission File Number: 000-15175 ADOBE INC. (Exact name of registrant as specified in its charter) Delaware 77-0019522 (State or other jurisdiction 


## 3. Corpus sanity checks

This is a quick quality check. Empty chunks are removed; this does not prove every OCR character is correct, so inspect examples and tables as part of evaluation.

In [21]:
chunk_df = pd.DataFrame(chunks)
print("Chunks by source page:", chunk_df["page"].nunique())
print("Empty chunks:", int((chunk_df["text"].str.strip() == "").sum()))
print("Words per chunk summary:")
display(chunk_df["text"].str.split().str.len().describe().to_frame("words"))
display(chunk_df[["chunk_id", "page", "text"]].head(3))

Chunks by source page: 93
Empty chunks: 0
Words per chunk summary:


,words
count,205.000000
mean,281.195122
std,95.525230
min,65.000000
25%,220.000000
50%,350.000000
75%,350.000000
max,350.000000


,chunk_id,page,text
0,0,1,UNITED STATES SECURITIES AND EXCHANGE COMMISSI...
1,1,1,of this chapter) during the preceding 12 month...
2,2,1,"$0.0001 par value per share, held by non-affil..."


## 4. Build the bi-encoder embeddings

Each corpus chunk is embedded once. At query time, we compute one query embedding and cosine similarities against the precomputed vectors. BGE models benefit from the retrieval query instruction below; MiniLM does not, and the code only uses it for BGE embedding models.

In [22]:
from sentence_transformers import SentenceTransformer, CrossEncoder

print("Loading bi-encoder:", EMBEDDING_MODEL_NAME)
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)

is_bge_embedder = "bge" in EMBEDDING_MODEL_NAME.lower() and "reranker" not in EMBEDDING_MODEL_NAME.lower()
QUERY_PREFIX = "Represent this sentence for searching relevant passages: " if is_bge_embedder else ""

chunk_texts = [c["text"] for c in chunks]
corpus_fingerprint = hashlib.sha256(
    (EMBEDDING_MODEL_NAME + "\n" + "\n".join(chunk_texts)).encode("utf-8")
).hexdigest()[:16]
embedding_cache_path = OUTPUT_DIR / f"embeddings_{corpus_fingerprint}.npy"

if embedding_cache_path.exists():
    embeddings = np.load(embedding_cache_path)
    print("Loaded cached embeddings:", embedding_cache_path)
else:
    embed_start = time.perf_counter()
    embeddings = embedder.encode(
        chunk_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    np.save(embedding_cache_path, embeddings)
    print(f"Encoded corpus in {time.perf_counter() - embed_start:.2f}s")

embeddings = np.asarray(embeddings, dtype=np.float32)
if embeddings.shape[0] != len(chunks):
    raise ValueError(f"Embedding count {embeddings.shape[0]} != chunk count {len(chunks)}")
if not np.isfinite(embeddings).all():
    raise ValueError("Embeddings contain NaN or infinite values")

print("Embedding shape:", embeddings.shape)
print("Normalized vector norm (first 5):", np.linalg.norm(embeddings[:5], axis=1).round(4))

Loading bi-encoder: BAAI/bge-base-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded cached embeddings: /kaggle/working/rag_outputs/embeddings_fd469b2b133eb72a.npy
Embedding shape: (205, 768)
Normalized vector norm (first 5): [1. 1. 1. 1. 1.]


## 5. Define dense retrieval and reranking

The cross-encoder is only applied to the dense Top 10. It cannot recover a chunk that wasn't retrieved into this candidate pool. This mirrors the standard two-stage retrieval architecture and keeps reranking computationally practical.

In [23]:
def dense_retrieve(query, k=TOP_K_CANDIDATES):
    """Return the dense Top-k chunks with cosine-similarity scores and timings."""
    start = time.perf_counter()
    query_embedding = embedder.encode(
        [QUERY_PREFIX + query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0]
    scores = embeddings @ query_embedding
    k = min(k, len(chunks))
    top_indices = np.argsort(scores)[::-1][:k]
    elapsed_ms = (time.perf_counter() - start) * 1000
    retrieved = []
    for rank, idx in enumerate(top_indices, start=1):
        item = dict(chunks[int(idx)])
        item.update({"dense_rank": rank, "dense_score": float(scores[idx])})
        retrieved.append(item)
    return retrieved, elapsed_ms


print("Loading cross-encoder:", RERANKER_MODEL_NAME)
reranker = CrossEncoder(RERANKER_MODEL_NAME, device=DEVICE)


def rerank_candidates(query, candidates):
    """Score the existing dense candidates and sort by cross-encoder score."""
    pairs = [(query, item["text"]) for item in candidates]
    start = time.perf_counter()
    scores = reranker.predict(pairs, show_progress_bar=False)
    elapsed_ms = (time.perf_counter() - start) * 1000
    reranked = []
    for item, score in zip(candidates, scores):
        copy_item = dict(item)
        copy_item["rerank_score"] = float(np.asarray(score).reshape(-1)[0])
        reranked.append(copy_item)
    reranked.sort(key=lambda item: item["rerank_score"], reverse=True)
    for rank, item in enumerate(reranked, start=1):
        item["rerank_rank"] = rank
    return reranked, elapsed_ms


# Smoke test: retrieval and reranking should both return results.
_smoke_query = "What is Adobe's strategy for generative AI?"
_smoke_dense, _ = dense_retrieve(_smoke_query)
_smoke_reranked, _ = rerank_candidates(_smoke_query, _smoke_dense)
print("Dense candidates:", len(_smoke_dense), "| Reranked candidates:", len(_smoke_reranked))
print("Dense Top 1:", _smoke_dense[0]["text"][:180])
print("Reranked Top 1:", _smoke_reranked[0]["text"][:180])

Loading cross-encoder: BAAI/bge-reranker-base


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Dense candidates: 10 | Reranked candidates: 10
Dense Top 1: AI Strategy AI represents a generational opportunity to serve an increasingly large and diverse customer universe. Adobe’s approach to AI is rooted in the belief that creativity is
Reranked Top 1: AI Strategy AI represents a generational opportunity to serve an increasingly large and diverse customer universe. Adobe’s approach to AI is rooted in the belief that creativity is


## 6. Evaluation queries

These queries cover strategy, cybersecurity, subscription revenue, business risks, financial performance, and legal/IP risks. Inspect the retrieved chunks; adjust or add questions if a query is ambiguous or poorly supported by the document.

In [24]:
EVAL_QUERIES = [
    "How does Adobe incorporate generative AI into its products and services?",
    "What cybersecurity risks does Adobe disclose, and how does the company manage them?",
    "What were Adobe's Digital Media and Digital Experience subscription revenues in fiscal 2025 compared with fiscal 2024?",
    "What major factors could adversely affect Adobe's business, financial condition, or results of operations?",
    "What changes does Adobe report in its financial performance for fiscal 2025?",
    "What legal proceedings and intellectual property risks does Adobe disclose?",
]

print("Number of evaluation queries:", len(EVAL_QUERIES))
for i, query in enumerate(EVAL_QUERIES, start=1):
    print(f"{i}. {query}")

Number of evaluation queries: 6
1. How does Adobe incorporate generative AI into its products and services?
2. What cybersecurity risks does Adobe disclose, and how does the company manage them?
3. What were Adobe's Digital Media and Digital Experience subscription revenues in fiscal 2025 compared with fiscal 2024?
4. What major factors could adversely affect Adobe's business, financial condition, or results of operations?
5. What changes does Adobe report in its financial performance for fiscal 2025?
6. What legal proceedings and intellectual property risks does Adobe disclose?


## 7. Run retrieval and compare rank shifts

This cell creates both Top 3 context sets for every query and records the position each dense candidate receives after reranking. Review the results rather than assuming reranking improves every question.

In [25]:
retrieval_results = {}
ranking_rows = []
latency_rows = []

for query_idx, query in enumerate(EVAL_QUERIES, start=1):
    dense, dense_ms = dense_retrieve(query, TOP_K_CANDIDATES)
    reranked, rerank_ms = rerank_candidates(query, dense)

    dense_by_id = {item["chunk_id"]: item for item in dense}
    reranked_by_id = {item["chunk_id"]: item for item in reranked}
    retrieval_results[query] = {
        "dense_top10": dense,
        "reranked_top10": reranked,
        "baseline_top3": dense[:TOP_K_CONTEXT],
        "reranked_top3": reranked[:TOP_K_CONTEXT],
    }

    for item in dense:
        rr_item = reranked_by_id[item["chunk_id"]]
        ranking_rows.append({
            "query_id": query_idx,
            "query": query,
            "chunk_id": item["chunk_id"],
            "page": item["page"],
            "dense_rank": item["dense_rank"],
            "dense_score": item["dense_score"],
            "rerank_rank": rr_item["rerank_rank"],
            "rerank_score": rr_item["rerank_score"],
            "rank_shift_positive_means_up": item["dense_rank"] - rr_item["rerank_rank"],
            "text_preview": item["text"][:300],
        })
    latency_rows.append({
        "query_id": query_idx,
        "query": query,
        "dense_retrieval_ms": dense_ms,
        "cross_encoder_rerank_ms": rerank_ms,
    })

rankings_df = pd.DataFrame(ranking_rows)
retrieval_latency_df = pd.DataFrame(latency_rows)

print("Per-query rank comparison (first 10 candidates shown per query):")
display(rankings_df.sort_values(["query_id", "rerank_rank"]))
print("Retrieval latency:")
display(retrieval_latency_df)

rankings_df.to_csv(OUTPUT_DIR / "retrieval_rankings.csv", index=False)
retrieval_latency_df.to_csv(OUTPUT_DIR / "retrieval_latency.csv", index=False)
print("Saved rankings and retrieval latency to", OUTPUT_DIR)

Per-query rank comparison (first 10 candidates shown per query):


,query_id,query,chunk_id,page,dense_rank,dense_score,rerank_rank,rerank_score,rank_shift_positive_means_up,text_preview
0,1,How does Adobe incorporate generative AI into ...,10,5,1,0.773499,1,0.448197,0,AI Strategy AI represents a generational oppor...
2,1,How does Adobe incorporate generative AI into ...,8,4,3,0.757582,2,0.290966,1,and creation to production and delivery. They ...
7,1,How does Adobe incorporate generative AI into ...,18,8,8,0.695791,3,0.220634,5,"including Adobe Photoshop, Adobe Illustrator a..."
4,1,How does Adobe incorporate generative AI into ...,11,5,5,0.720873,4,0.150798,1,"making, optimizing workflows and enabling real..."
1,1,How does Adobe incorporate generative AI into ...,19,8,2,0.764352,5,0.101348,-3,own aesthetic style through Firefly generative...
6,1,How does Adobe incorporate generative AI into ...,5,3,7,0.703686,6,0.034757,1,"66 Id 66 Id 66 “Adobe,” “Acrobat,” “Photoshop,..."
3,1,How does Adobe incorporate generative AI into ...,9,4,4,0.724469,7,0.020998,-3,Adobe Experience Platform AI Assistant. Adobe ...
5,1,How does Adobe incorporate generative AI into ...,90,34,6,0.704753,8,0.007053,-2,Spanning both our Digital Media and Digital Ex...
9,1,How does Adobe incorporate generative AI into ...,14,6,10,0.692029,9,0.004186,1,productivity solutions that allows users to co...
8,1,How does Adobe incorporate generative AI into ...,20,9,9,0.695650,10,0.001307,-1,Adobe Agent Orchestrator Adobe Experience Plat...


Retrieval latency:


,query_id,query,dense_retrieval_ms,cross_encoder_rerank_ms
0,1,How does Adobe incorporate generative AI into ...,13.848547,299.108734
1,2,"What cybersecurity risks does Adobe disclose, ...",11.876708,300.164903
2,3,What were Adobe's Digital Media and Digital Ex...,11.393866,299.610880
3,4,What major factors could adversely affect Adob...,11.296279,305.865837
4,5,What changes does Adobe report in its financia...,11.886457,298.655669
5,6,What legal proceedings and intellectual proper...,11.601278,305.097704


Saved rankings and retrieval latency to /kaggle/working/rag_outputs


## 8. Prepare the same prompt for both pipelines

Only the retrieved Top 3 evidence changes. The system instruction and generation settings remain identical. Each context includes its source page so the answer can cite the document.

In [26]:
SYSTEM_INSTRUCTION = (
    "You are a careful assistant answering questions about Adobe Inc.'s fiscal 2025 Form 10-K. "
    "Use only the supplied context. If the context does not contain the answer, say that the "
    "provided excerpts do not establish it. Do not invent facts or numbers. Cite relevant PDF "
    "page numbers in parentheses, for example (p. 42). Distinguish reported facts from risks or projections."
)


def build_prompt(query, context_chunks):
    context_parts = []
    for item in context_chunks:
        context_parts.append(
            f"[Source: PDF page {item['page']}; chunk {item['chunk_id']}]\n{item['text']}"
        )
    context = "\n\n---\n\n".join(context_parts)
    return (
        f"{SYSTEM_INSTRUCTION}\n\n"
        f"Question: {query}\n\n"
        f"Context excerpts (retrieved from the Adobe 2025 Form 10-K):\n{context}\n\n"
        "Answer the question directly and concisely, using evidence from the excerpts. "
        "Cite PDF page numbers for important claims."
    )


# Visually confirm the prompt structure before generation.
_example_query = EVAL_QUERIES[0]
print(build_prompt(_example_query, retrieval_results[_example_query]["baseline_top3"])[:2500])

You are a careful assistant answering questions about Adobe Inc.'s fiscal 2025 Form 10-K. Use only the supplied context. If the context does not contain the answer, say that the provided excerpts do not establish it. Do not invent facts or numbers. Cite relevant PDF page numbers in parentheses, for example (p. 42). Distinguish reported facts from risks or projections.

Question: How does Adobe incorporate generative AI into its products and services?

Context excerpts (retrieved from the Adobe 2025 Form 10-K):
[Source: PDF page 5; chunk 10]
AI Strategy AI represents a generational opportunity to serve an increasingly large and diverse customer universe. Adobe’s approach to AI is rooted in the belief that creativity is a uniquely human trait — and that AI has the power to assist and amplify human ingenuity and enhance productivity. Adobe’s AI leadership builds on our expertise across core creative categories like imaging, design, illustration and video as well as digital documents and d

## 9. Load a local instruction-tuned generator (no API key)

This section loads Gemma 3 4B from Hugging Face using `transformers`. No Gemini API key or paid generation endpoint is used. Kaggle Internet must be enabled for the first model download, and the model's Hugging Face access/usage conditions may need to be accepted beforehand.

The default attempts 4-bit quantization to reduce VRAM usage. If `bitsandbytes` or 4-bit loading is unsupported in the current runtime, the next cell falls back to FP16. If FP16 does not fit, switch to a smaller instruction model or use a Kaggle accelerator with more VRAM.

In [27]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

print("Hugging Face authentication completed.")

Hugging Face authentication completed.


In [28]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

generator = None
tokenizer = None

print("Loading local generator:", GENERATION_MODEL_NAME)
print("GPU available:", torch.cuda.is_available())

if not RUN_GENERATION:
    print("Generation disabled by configuration.")
else:
    tokenizer = AutoTokenizer.from_pretrained(
        GENERATION_MODEL_NAME,
        use_fast=True,
    )

    if LOAD_GENERATOR_IN_4BIT and torch.cuda.is_available():
        try:
            from transformers import BitsAndBytesConfig

            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )
            generator_model = AutoModelForCausalLM.from_pretrained(
                GENERATION_MODEL_NAME,
                device_map="auto",
                quantization_config=quantization_config,
                torch_dtype=torch.float16,
            )
            print("Loaded generator in 4-bit mode.")
        except Exception as exc:
            print("4-bit load failed; attempting FP16. Reason:", repr(exc))
            gc.collect()
            torch.cuda.empty_cache()
            generator_model = AutoModelForCausalLM.from_pretrained(
                GENERATION_MODEL_NAME,
                device_map="auto",
                torch_dtype=torch.float16,
            )
    else:
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        generator_model = AutoModelForCausalLM.from_pretrained(
            GENERATION_MODEL_NAME,
            device_map="auto" if torch.cuda.is_available() else None,
            torch_dtype=dtype,
        )

    generator = pipeline(
        "text-generation",
        model=generator_model,
        tokenizer=tokenizer,
    )
    print("Local generator loaded successfully.")


Loading local generator: google/gemma-3-4b-it
GPU available: True


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Loaded generator in 4-bit mode.
Local generator loaded successfully.


## 10. Generate comparable answers and profile end-to-end latency

Two local generation calls are made per evaluation query: one for the dense Top 3 and one for the reranked Top 3. Both use the same Gemma model, prompt template, greedy decoding (`do_sample=False`), and maximum new-token limit. The only intended difference is the retrieved context.

In [29]:
def generate_answer(prompt):
    """Generate locally with fixed settings; return answer text and latency."""
    if generator is None:
        raise RuntimeError("Generator is not loaded. Check Section 9 and RUN_GENERATION.")

    start = time.perf_counter()
    outputs = generator(
        prompt,
        max_new_tokens=GENERATION_MAX_NEW_TOKENS,
        do_sample=False,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    elapsed_ms = (time.perf_counter() - start) * 1000
    answer = outputs[0]["generated_text"].strip()
    return answer, elapsed_ms


generation_rows = []
generation_latencies = []

if RUN_GENERATION and generator is not None:
    queries_for_generation = EVAL_QUERIES[:MAX_GENERATION_QUERIES]
    for query_idx, query in enumerate(queries_for_generation, start=1):
        result = retrieval_results[query]
        print(f"Generating local answers for query {query_idx}/{len(queries_for_generation)}")

        baseline_prompt = build_prompt(query, result["baseline_top3"])
        baseline_answer, baseline_generation_ms = generate_answer(baseline_prompt)
        reranked_prompt = build_prompt(query, result["reranked_top3"])
        reranked_answer, reranked_generation_ms = generate_answer(reranked_prompt)

        generation_rows.extend([
            {
                "query_id": query_idx,
                "query": query,
                "pipeline": "bi_encoder_top3",
                "context_chunk_ids": [x["chunk_id"] for x in result["baseline_top3"]],
                "context_pages": [x["page"] for x in result["baseline_top3"]],
                "answer": baseline_answer,
                "generation_ms": baseline_generation_ms,
            },
            {
                "query_id": query_idx,
                "query": query,
                "pipeline": "cross_encoder_reranked_top3",
                "context_chunk_ids": [x["chunk_id"] for x in result["reranked_top3"]],
                "context_pages": [x["page"] for x in result["reranked_top3"]],
                "answer": reranked_answer,
                "generation_ms": reranked_generation_ms,
            },
        ])
        generation_latencies.extend([
            {"query_id": query_idx, "query": query, "pipeline": "bi_encoder_top3", "generation_ms": baseline_generation_ms},
            {"query_id": query_idx, "query": query, "pipeline": "cross_encoder_reranked_top3", "generation_ms": reranked_generation_ms},
        ])
        print(f"  baseline {baseline_generation_ms:.0f} ms | reranked {reranked_generation_ms:.0f} ms")
else:
    print("Generation skipped because RUN_GENERATION=False or the local generator is unavailable.")

generation_df = pd.DataFrame(generation_rows)
generation_latency_df = pd.DataFrame(generation_latencies)
if not generation_df.empty:
    generation_df.to_csv(OUTPUT_DIR / "generation_comparison.csv", index=False)
    display(generation_df[["query_id", "pipeline", "context_pages", "generation_ms", "answer"]])
    print("Saved generation comparison.")


Generating local answers for query 1/6
  baseline 43223 ms | reranked 42924 ms
Generating local answers for query 2/6
  baseline 42523 ms | reranked 42587 ms
Generating local answers for query 3/6
  baseline 42396 ms | reranked 43039 ms
Generating local answers for query 4/6
  baseline 42705 ms | reranked 42572 ms
Generating local answers for query 5/6


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  baseline 42948 ms | reranked 42835 ms
Generating local answers for query 6/6
  baseline 42921 ms | reranked 42670 ms


,query_id,pipeline,context_pages,generation_ms,answer
0,1,bi_encoder_top3,"[5, 8, 4]",43223.272097,
1,1,cross_encoder_reranked_top3,"[5, 4, 8]",42924.207294,
2,2,bi_encoder_top3,"[29, 29, 29]",42523.159218,
3,2,cross_encoder_reranked_top3,"[29, 29, 2]",42587.303084,
4,3,bi_encoder_top3,"[33, 36, 33]",42396.233194,
5,3,cross_encoder_reranked_top3,"[33, 35, 37]",43038.822749,
6,4,bi_encoder_top3,"[52, 88, 17]",42704.790329,
7,4,cross_encoder_reranked_top3,"[81, 2, 17]",42571.561832,
8,5,bi_encoder_top3,"[35, 59, 33]",42947.724276,
9,5,cross_encoder_reranked_top3,"[35, 33, 68]",42834.818157,


Saved generation comparison.


## 11. Analyze whether reranking actually helps

Do not label a query a “reranking success” just because the rankings differ. Inspect the chunks and answers. A useful failure case is one where the dense Top 3 misses or poorly represents the evidence, while a relevant chunk outside the dense Top 3 is promoted into the reranked Top 3 and supports a better answer.

In [30]:
def show_query_comparison(query):
    """Display Top 3 context before/after reranking plus any generated answers."""
    result = retrieval_results[query]
    print("QUESTION:", query)
    print("\n=== Dense bi-encoder Top 3 ===")
    for rank, item in enumerate(result["baseline_top3"], start=1):
        print(f"\nRank {rank} | page {item['page']} | chunk {item['chunk_id']} | dense={item['dense_score']:.4f}")
        print(item["text"][:700])
    print("\n=== Cross-encoder reranked Top 3 ===")
    for rank, item in enumerate(result["reranked_top3"], start=1):
        print(f"\nRank {rank} | page {item['page']} | chunk {item['chunk_id']} | rerank={item['rerank_score']:.4f} | dense rank={item['dense_rank']}")
        print(item["text"][:700])
    if not generation_df.empty:
        selected = generation_df[generation_df["query"] == query]
        for _, row in selected.iterrows():
            print(f"\n=== {row['pipeline']} answer ===\n{row['answer']}")


# Inspect these queries, then use the evidence to choose at least two failure cases
# where the reranked Top 3 is more useful than the dense Top 3.
show_query_comparison(EVAL_QUERIES[0])
show_query_comparison(EVAL_QUERIES[2])

QUESTION: How does Adobe incorporate generative AI into its products and services?

=== Dense bi-encoder Top 3 ===

Rank 1 | page 5 | chunk 10 | dense=0.7735
AI Strategy AI represents a generational opportunity to serve an increasingly large and diverse customer universe. Adobe’s approach to AI is rooted in the belief that creativity is a uniquely human trait — and that AI has the power to assist and amplify human ingenuity and enhance productivity. Adobe’s AI leadership builds on our expertise across core creative categories like imaging, design, illustration and video as well as digital documents and digital experiences. Over the past several years, we have incorporated AI into our solutions and delivered hundreds of AI capabilities to enable customers to create, work and collaborate more efficiently. To address our customers’ need for a holist

Rank 2 | page 8 | chunk 19 | dense=0.7644
own aesthetic style through Firefly generative AI image models that are trained with an enterprise

## 12. Rank movement and top-3 overlap summary

A positive shift means a candidate moved upward after reranking (e.g., dense rank 8 to reranked rank 2 gives +6). Top-3 overlap measures how much the selected evidence changes; a change alone is not necessarily an improvement.

In [31]:
rank_shift_summary = (
    rankings_df.assign(
        moved_up=rankings_df["rank_shift_positive_means_up"] > 0,
        moved_down=rankings_df["rank_shift_positive_means_up"] < 0,
    )
    .groupby(["query_id", "query"], as_index=False)
    .agg(
        candidates_scored=("chunk_id", "count"),
        average_dense_rank=("dense_rank", "mean"),
        average_reranked_rank=("rerank_rank", "mean"),
        candidates_moved_up=("moved_up", "sum"),
        candidates_moved_down=("moved_down", "sum"),
        max_upward_shift=("rank_shift_positive_means_up", "max"),
    )
)

overlap_rows = []
for i, query in enumerate(EVAL_QUERIES, start=1):
    result = retrieval_results[query]
    dense_ids = {x["chunk_id"] for x in result["baseline_top3"]}
    reranked_ids = {x["chunk_id"] for x in result["reranked_top3"]}
    overlap_rows.append({
        "query_id": i,
        "query": query,
        "top3_shared_chunks": len(dense_ids & reranked_ids),
        "top3_changed_chunks": len(dense_ids ^ reranked_ids) // 2,
        "top3_overlap_fraction": len(dense_ids & reranked_ids) / max(1, len(dense_ids | reranked_ids)),
    })

overlap_df = pd.DataFrame(overlap_rows)
display(rank_shift_summary)
display(overlap_df)
rank_shift_summary.to_csv(OUTPUT_DIR / "rank_shift_summary.csv", index=False)
overlap_df.to_csv(OUTPUT_DIR / "top3_overlap.csv", index=False)

,query_id,query,candidates_scored,average_dense_rank,average_reranked_rank,candidates_moved_up,candidates_moved_down,max_upward_shift
0,1,How does Adobe incorporate generative AI into ...,10,5.5,5.5,5,4,5
1,2,"What cybersecurity risks does Adobe disclose, ...",10,5.5,5.5,4,3,3
2,3,What were Adobe's Digital Media and Digital Ex...,10,5.5,5.5,5,4,3
3,4,What major factors could adversely affect Adob...,10,5.5,5.5,4,3,6
4,5,What changes does Adobe report in its financia...,10,5.5,5.5,4,4,4
5,6,What legal proceedings and intellectual proper...,10,5.5,5.5,5,5,7


,query_id,query,top3_shared_chunks,top3_changed_chunks,top3_overlap_fraction
0,1,How does Adobe incorporate generative AI into ...,2,1,0.5
1,2,"What cybersecurity risks does Adobe disclose, ...",2,1,0.5
2,3,What were Adobe's Digital Media and Digital Ex...,1,2,0.2
3,4,What major factors could adversely affect Adob...,1,2,0.2
4,5,What changes does Adobe report in its financia...,2,1,0.5
5,6,What legal proceedings and intellectual proper...,0,3,0.0


## 13. Latency summary

The embedding time is measured once for the whole corpus (unless loaded from cache). For each query, compare dense retrieval time with cross-encoder reranking time. Generation latency is profiled separately because API/network latency can dominate end-to-end time.

In [32]:
latency_summary = retrieval_latency_df.copy()
if not generation_latency_df.empty:
    latency_pivot = generation_latency_df.pivot(index="query_id", columns="pipeline", values="generation_ms").reset_index()
    latency_summary = latency_summary.merge(latency_pivot, on="query_id", how="left")

print("Mean dense retrieval (ms):", round(retrieval_latency_df["dense_retrieval_ms"].mean(), 2))
print("Mean cross-encoder reranking (ms):", round(retrieval_latency_df["cross_encoder_rerank_ms"].mean(), 2))
if not generation_latency_df.empty:
    print("Mean baseline generation (ms):", round(generation_latency_df.loc[generation_latency_df.pipeline == "bi_encoder_top3", "generation_ms"].mean(), 2))
    print("Mean reranked generation (ms):", round(generation_latency_df.loc[generation_latency_df.pipeline == "cross_encoder_reranked_top3", "generation_ms"].mean(), 2))

display(latency_summary)
latency_summary.to_csv(OUTPUT_DIR / "latency_summary.csv", index=False)

Mean dense retrieval (ms): 11.98
Mean cross-encoder reranking (ms): 301.42
Mean baseline generation (ms): 42786.03
Mean reranked generation (ms): 42771.18


,query_id,query,dense_retrieval_ms,cross_encoder_rerank_ms,bi_encoder_top3,cross_encoder_reranked_top3
0,1,How does Adobe incorporate generative AI into ...,13.848547,299.108734,43223.272097,42924.207294
1,2,"What cybersecurity risks does Adobe disclose, ...",11.876708,300.164903,42523.159218,42587.303084
2,3,What were Adobe's Digital Media and Digital Ex...,11.393866,299.610880,42396.233194,43038.822749
3,4,What major factors could adversely affect Adob...,11.296279,305.865837,42704.790329,42571.561832
4,5,What changes does Adobe report in its financia...,11.886457,298.655669,42947.724276,42834.818157
5,6,What legal proceedings and intellectual proper...,11.601278,305.097704,42921.028005,42670.382429


## 14. Save full results

These artefacts are saved to `/kaggle/working/rag_outputs`. Use the Kaggle Output panel to download them, or create a ZIP below.

In [33]:
# Save the complete retrieval and answer records as JSON for reproducibility.
full_payload = {
    "config": {
        "embedding_model": EMBEDDING_MODEL_NAME,
        "reranker_model": RERANKER_MODEL_NAME,
        "generation_model": GENERATION_MODEL_NAME,
        "generation_mode": "local_transformers",
        "generation_max_new_tokens": GENERATION_MAX_NEW_TOKENS,
        "top_k_candidates": TOP_K_CANDIDATES,
        "top_k_context": TOP_K_CONTEXT,
        "generation_temperature": GENERATION_TEMPERATURE,
        "generation_max_output_tokens": GENERATION_MAX_OUTPUT_TOKENS,
    },
    "queries": [],
}

for query_idx, query in enumerate(EVAL_QUERIES, start=1):
    rr = retrieval_results[query]
    entry = {
        "query_id": query_idx,
        "query": query,
        "dense_top10": rr["dense_top10"],
        "reranked_top10": rr["reranked_top10"],
        "baseline_top3_chunk_ids": [x["chunk_id"] for x in rr["baseline_top3"]],
        "reranked_top3_chunk_ids": [x["chunk_id"] for x in rr["reranked_top3"]],
    }
    if not generation_df.empty:
        entry["answers"] = generation_df[generation_df["query"] == query].to_dict(orient="records")
    full_payload["queries"].append(entry)

with open(OUTPUT_DIR / "full_pipeline_results.json", "w", encoding="utf-8") as f:
    json.dump(full_payload, f, ensure_ascii=False, indent=2, default=str)

import shutil
zip_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("\nSaved files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"- {path.name}: {path.stat().st_size / 1024:.1f} KB")
print("\nZIP bundle:", zip_path)

NameError: name 'GENERATION_TEMPERATURE' is not defined

## 15. Report notes: items to record after inspecting outputs

Do not fabricate improvements. Complete these from the results you observe:

- **Corpus:** 93-page Adobe 2025 Form 10-K OCR extraction; report actual chunk count and approximate total words printed in Section 2.
- **Models:** record the actual bi-encoder and cross-encoder IDs from the configuration.
- **Baseline:** dense cosine similarity over all chunk embeddings; retrieve Top 10, pass dense Top 3 to the LLM.
- **Reranked pipeline:** rerank only the same dense Top 10, pass cross-encoder Top 3 to the same LLM.
- **At least two failure cases:** select queries only after inspecting Section 11. For each, record the dense rank of a relevant chunk, its reranked rank, the page number, and how the answer changed. If reranking does not help a query, report that honestly and select another query.
- **Latency:** report mean dense retrieval, reranking, and generation latency separately. State that reranking cannot introduce documents absent from the initial Top 10.
- **Limitations:** OCR errors and flattened financial tables can affect retrieval; a single document and a small query set do not establish universal superiority.

Keep the code and outputs reproducible. Do not commit API keys or other secrets.